In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# R04 — Train a representation and evaluate frozen probes

Read the single primary lesson before this lab. Predict each result. Numerical correctness is not independent mastery. See `representations/SOURCE_AUDIT.md` for provenance.

In [2]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
here = Path.cwd().resolve()
ROOT = next((p for p in [here, *here.parents] if (p / 'src/shape_lab').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook within the extracted course project.')
sys.path.insert(0, str(ROOT / 'src'))
from shape_lab import representations as r
from shape_lab import representation_experiments as exp
print('Course source:', ROOT)


Course source: /tmp/shape-course-q90rdcpz/course


## Train both fixed-seed methods and retain every baseline result. This uses the historical exposed course split.

In [3]:
result=exp.train_digits(epochs=25,seeds=(11,23))
rows=pd.DataFrame([{k:v for k,v in row.items() if k!='embedding_train'} for row in result['models']])
print(rows.to_string(index=False))
assert len(rows)==9
assert result['protocol']['pretraining_labels_used'] is False
assert rows.test_n.nunique()==1

                     model  seed  probe_C  validation_accuracy  test_correct  test_n  test_accuracy
                raw_pixels     0      1.0             0.963889           346     360       0.961111
                     pca12     0      1.0             0.922222           335     360       0.930556
label_independent_gaussian    80      0.1             0.108333            39     360       0.108333
            random_encoder    11      1.0             0.783333           287     360       0.797222
            agreement_only    11      0.1             0.483333           196     360       0.544444
          agreement_sigreg    11     10.0             0.886111           323     360       0.897222
            random_encoder    23      1.0             0.800000           289     360       0.802778
            agreement_only    23      0.1             0.466667           190     360       0.527778
          agreement_sigreg    23     10.0             0.888889           325     360       0.902778


## Inspect output scale as well as effective rank.

In [4]:
geometry=pd.DataFrame([{'model':m['model'],'seed':m['seed'],**m['embedding_train']} for m in result['models']])
print(geometry[['model','seed','mean_coordinate_std','effective_rank']].to_string(index=False))
plt.figure(figsize=(9,5)); plt.bar([f"{m['model']} / {m['seed']}" for m in result['models']], [m['test_accuracy'] for m in result['models']]); plt.xticks(rotation=55,ha='right'); plt.ylabel('Test accuracy'); plt.title('One exposed demonstration partition; not a universal ranking'); plt.tight_layout(); plt.show()

                     model  seed  mean_coordinate_std  effective_rank
                raw_pixels     0             0.230184       20.509041
                     pca12     0             0.525670        9.772061
label_independent_gaussian    80             0.996223       11.923550
            random_encoder    11             0.046675        7.885716
            agreement_only    11             0.001459        9.884147
          agreement_sigreg    11             1.018692        4.716825
            random_encoder    23             0.046599        7.191061
            agreement_only    23             0.001415        9.607549
          agreement_sigreg    23             0.974516        4.551108


/tmp/ipykernel_6944/1629870270.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.figure(figsize=(9,5)); plt.bar([f"{m['model']} / {m['seed']}" for m in result['models']], [m['test_accuracy'] for m in result['models']]); plt.xticks(rotation=55,ha='right'); plt.ylabel('Test accuracy'); plt.title('One exposed demonstration partition; not a universal ranking'); plt.tight_layout(); plt.show()


## Inspect optimization history separately from downstream accuracy.

In [5]:
history=pd.read_csv(ROOT/'reports/representations/r04/training_history.csv')
plt.figure(figsize=(8,4))
for (method,seed),group in history.groupby(['model','seed']):
    plt.plot(group.epoch,group.agreement,label=f'{method} / {seed}')
plt.yscale('log'); plt.xlabel('Epoch'); plt.ylabel('Agreement loss'); plt.legend(); plt.tight_layout(); plt.show()
print('Artifacts:', ROOT/'reports/representations/r04')

Artifacts: /tmp/shape-course-q90rdcpz/course/reports/representations/r04


/tmp/ipykernel_6944/3178451812.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.yscale('log'); plt.xlabel('Epoch'); plt.ylabel('Agreement loss'); plt.legend(); plt.tight_layout(); plt.show()


## Read sample-level errors. Labels are used here for evaluation, not pretraining.

In [6]:
predictions=pd.read_csv(ROOT/'reports/representations/r04/predictions.csv')
print(predictions[predictions.truth!=predictions.prediction].head(12).to_string(index=False))
assert predictions.groupby(['model','seed']).sample_id.nunique().eq(360).all()

     model  seed  sample_id  truth  prediction
raw_pixels     0         77      2           1
raw_pixels     0        363      1           6
raw_pixels     0        539      3           8
raw_pixels     0        578      3           8
raw_pixels     0        607      3           8
raw_pixels     0        723      2           8
raw_pixels     0        769      8           2
raw_pixels     0        890      8           1
raw_pixels     0       1095      4           9
raw_pixels     0       1149      8           1
raw_pixels     0       1256      1           6
raw_pixels     0       1690      3           8


## Your evidence
Record: observation unit; assumptions; prediction; calculated result; difference from your prediction; and one unsupported conclusion. Attempt the separate learner notebook before opening the reference solutions.